# 롤백과 복구 검증

두 리비전을 배포한 뒤 경보를 근거로 이전 다이제스트를 다시 활성화합니다.

## 코드 읽기와 개념 경계

롤백은 이전 다이제스트를 다시 활성화하는 변경이고, 복구 검증은 그 뒤 서비스가 기대 상태로 돌아왔는지 확인하는 별도 단계입니다. 예제는 v1과 v2 배포 사건을 기록하고 경보 프로세스의 실패를 확인한 뒤 `rollback.py`로 새 사건을 추가합니다. 마지막에는 활성 다이제스트와 HTTP 검사를 모두 확인합니다.

이전 이미지로 돌아갔다고 데이터베이스와 외부 작업까지 되돌아가는 것은 아닙니다. 스키마가 비호환이거나 이미 발송된 메시지가 있다면 코드 롤백만으로 복구되지 않습니다. 배포 전에는 이전 버전이 현재 데이터 구조를 읽을 수 있는지, 롤백 후 어떤 대표 요청으로 기능을 확인할지 정합니다.

In [1]:
from pathlib import Path
import hashlib, subprocess, sys, tempfile, json, shutil
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()
v1 = "sha256:" + hashlib.sha256(b"stable-v1").hexdigest()
v2 = "sha256:" + hashlib.sha256(b"regressed-v2").hexdigest()

In [2]:
temp = tempfile.TemporaryDirectory()
t = Path(temp.name)
shutil.copytree(lab / "scripts", t / "scripts")
(t / "runs").mkdir()
for digest in (v1, v2):
    result = subprocess.run([sys.executable, "scripts/deploy.py", digest], cwd=t, text=True, capture_output=True)
    print(result.stdout.strip())
print(json.loads((t / "runs/deployment.json").read_text()))

deployed revision=1 digest=sha256:fcb67e1b039a...
deployed revision=2 digest=sha256:9e909250ecb4...
[{'revision': 1, 'event': 'deploy', 'digest': 'sha256:fcb67e1b039aad4c998a635c47ffe5d09eacb6bbccc98acbdeed863861729307', 'status': 'ready'}, {'revision': 2, 'event': 'deploy', 'digest': 'sha256:9e909250ecb485a21861853887a9ffacd61dbf3af37eb93481bfb91ca3244f4e', 'status': 'ready'}]


## 실패 재현

v2 배포 후의 고정 운영 자료에서 경보 프로세스는 종료 코드 1을 반환합니다.

In [3]:
failed = subprocess.run([sys.executable, str(lab / "scripts/check_metrics.py")], text=True, capture_output=True)
print(failed.stdout.strip().splitlines()[-1], "exit_code=", failed.returncode)
assert failed.returncode == 1

alert: FIRING longest_streak=2 exit_code= 1


## 수정

롤백 스크립트로 새 롤백 사건을 추가하고 활성 다이제스트가 v1인지 확인합니다.

In [4]:
rolled = subprocess.run([sys.executable, "scripts/rollback.py"], cwd=t, text=True, capture_output=True)
history = json.loads((t / "runs/deployment.json").read_text())
print(rolled.stdout.strip(), "active=", history[-1]["digest"][:19])
http = subprocess.run([sys.executable, str(lab / "scripts/verify_http_rollback.py")], text=True, capture_output=True)
print(http.stdout.strip())
assert rolled.returncode == 0 and history[-1]["digest"] == v1 and http.returncode == 0
temp.cleanup()

rolled back 2 -> 1 active= sha256:fcb67e1b039a


200 v1 ok
503 v2 regressed
200 v1 ok
